## K-Nearest Neighbors

In [15]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [16]:
df = pd.read_csv("data/algerian_forest_fires_proj4.csv")
df.head()

,region,temp_c,rel_humidity_percent,wind_speed_kmh,rainfall_mm,ffmc,dmc,dc,isi,bui,fwi,classes
0,Bejaia,29.0,57.0,18.0,0.0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
1,Bejaia,29.0,61.0,13.0,1.3,64.4,4.1,7.6,1.0,3.9,0.4,not fire
2,Bejaia,26.0,82.0,22.0,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
3,Bejaia,25.0,89.0,13.0,2.5,28.6,1.3,6.9,0.0,1.7,0.0,not fire
4,Bejaia,27.0,77.0,16.0,0.0,64.8,3.0,14.2,1.2,3.9,0.5,not fire


In [17]:
df.shape

(243, 12)

### Creating the KNN

In [18]:
tested_df = df.drop('region', axis=1).copy()
tested_df.head()

,temp_c,rel_humidity_percent,wind_speed_kmh,rainfall_mm,ffmc,dmc,dc,isi,bui,fwi,classes
0,29.0,57.0,18.0,0.0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
1,29.0,61.0,13.0,1.3,64.4,4.1,7.6,1.0,3.9,0.4,not fire
2,26.0,82.0,22.0,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
3,25.0,89.0,13.0,2.5,28.6,1.3,6.9,0.0,1.7,0.0,not fire
4,27.0,77.0,16.0,0.0,64.8,3.0,14.2,1.2,3.9,0.5,not fire


In [19]:
fire_data_x = tested_df.drop('classes', axis=1)
fire_data_y = tested_df['classes']

cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [20]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

parameters = {
    'knn__n_neighbors': range(1, 26),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['manhattan', 'euclidean']
}

In [21]:
knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=parameters,
    scoring='accuracy',
    cv=cross_validation,
    n_jobs=-1,
    verbose=2
)
knn_grid.fit(fire_data_x, fire_data_y)
print('Best parameters ', knn_grid.best_params_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=distance; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=distance; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=1, knn__weights=distance; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=2, knn__weights=uniform; total time=   0.0s
[CV]

In [22]:
BEST_KNN_CLF = knn_grid.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",19
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",3

In [23]:
final_scores = cross_val_score(
    BEST_KNN_CLF,
    fire_data_x,
    fire_data_y,
    scoring='accuracy',
    cv=cross_validation,
)
print(final_scores)
print(final_scores.mean())
print(final_scores.std())

[0.89795918 0.95918367 0.95918367 0.9375     0.95833333]
0.9424319727891156
0.02373286980471338


In [ ]:
knn_predict_y = cross_val_predict(BEST_KNN_CLF, fire_data_x, fire_data_y, cv=cross_validation)

In [ ]:
classification_accuracy = round(accuracy_score(fire_data_y, knn_predict_y) * 100, 4)
confusion_matrix = confusion_matrix(fire_data_y, knn_predict_y)
per_class_classification = classification_report(fire_data_y, knn_predict_y)